In [46]:
!pip install /kaggle/input/datasets/thanhduc1108/vlsp2025-kd-wheels/huggingface_hub-1.11.0-py3-none-any.whl --no-deps --force-reinstall
!pip install /kaggle/input/datasets/thanhduc1108/vlsp2025-kd-wheels/transformers-5.5.4-py3-none-any.whl --no-deps --force-reinstall

Processing /kaggle/input/vlsp2025-kd-wheels/huggingface_hub-1.11.0-py3-none-any.whl
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
Processing /kaggle/input/vlsp2025-kd-wheels/transformers-5.5.4-py3-none-any.whl
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.4
    Uninstalling transformers-5.5.4:
      Successfully uninstalled transformers-5.5.4


In [47]:
import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.5.4


In [48]:
!pip install transformers accelerate bitsandbytes peft trl datasets \
    --no-index \
    --find-links=/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-wheels/ \
    -U

Looking in links: /kaggle/input/vlsp2025-kd-wheels/
Processing /kaggle/input/vlsp2025-kd-wheels/accelerate-1.13.0-py3-none-any.whl
Processing /kaggle/input/vlsp2025-kd-wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
Processing /kaggle/input/vlsp2025-kd-wheels/peft-0.19.1-py3-none-any.whl
Processing /kaggle/input/vlsp2025-kd-wheels/trl-1.2.0-py3-none-any.whl
Processing /kaggle/input/vlsp2025-kd-wheels/datasets-4.8.4-py3-none-any.whl
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
Processing /kaggle/input/vlsp2025-kd-wheels/transformers-5.5.4-py3-none-any.whl
ERROR: Could not find a version that satisfies the requirement hf-xet<2.0.0,>=1.4.3; platform_machine == "x86_64" or platform_machine == "amd64" or platform_machine == "AMD64" or platform_machine == "arm64" or platform_machine == "aarch64" (from huggingface-hub) (from versions: none)
ERROR: No matching distribution f

In [49]:
import os
from pathlib import Path

REQUIRED_INPUTS = {
    "/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-pipeline":   "Pipeline code   (Dataset: thanhduc1108/vlsp2025-kd-pipeline)",
    "/kaggle/input/datasets/thanhduc1108/finqa-en":               "FinQA dataset   (Dataset: thanhduc1108/finqa-en)",
    "/kaggle/input/datasets/thanhduc1108/vinumericalqa-private":  "ViNumQA dataset (Dataset: thanhduc1108/vinumericalqa-private)",
}
OPTIONAL_INPUTS = {
    "/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-wheels":     "Offline wheels  (Dataset: thanhduc1108/vlsp2025-kd-wheels)",
}

print("=" * 65)
print("CELL 0: Pre-flight Input Checklist")
print("=" * 65)
all_ok = True
for path, label in REQUIRED_INPUTS.items():
    exists = os.path.exists(path)
    status = "✓ FOUND  " if exists else "✗ MISSING"
    print(f"  [{status}] {label}")
    print(f"            path: {path}")
    if not exists:
        all_ok = False

for path, label in OPTIONAL_INPUTS.items():
    exists = os.path.exists(path)
    status = "✓ found  " if exists else "○ absent "
    print(f"  [{status}] {label}  (optional)")

print()
if not all_ok:
    print("STOP: Add the MISSING datasets above before continuing.")
    print("  Notebook -> top-right panel -> '+ Add Data' -> search by dataset name")
    raise SystemExit("Missing required input datasets. Add them first, then re-run.")
else:
    print("All required inputs present. Proceed to Cell 1.")

import subprocess, sys
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\nGPU: {result.stdout.strip()}")
else:
    print("\nWARNING: nvidia-smi not found. Enable GPU: Notebook -> Settings -> Accelerator -> GPU")

CELL 0: Pre-flight Input Checklist
  [✓ FOUND  ] Pipeline code   (Dataset: thanhduc1108/vlsp2025-kd-pipeline)
            path: /kaggle/input/vlsp2025-kd-pipeline
  [✓ FOUND  ] FinQA dataset   (Dataset: thanhduc1108/finqa-en)
            path: /kaggle/input/finqa-en
  [✓ FOUND  ] ViNumQA dataset (Dataset: thanhduc1108/vinumericalqa-private)
            path: /kaggle/input/vinumericalqa-private
  [✓ found  ] Offline wheels  (Dataset: thanhduc1108/vlsp2025-kd-wheels)  (optional)

All required inputs present. Proceed to Cell 1.

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


## CELL 1: Install Dependencies

In [50]:
import subprocess, os, sys
from pathlib import Path

WHEELS_DIR = "/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-wheels"

print("=" * 60)
print("🚀 BẮT ĐẦU CÀI ĐẶT THƯ VIỆN OFFLINE (Bypass Dependencies)")
print("=" * 60)

if os.path.exists(WHEELS_DIR):
    wheel_paths = list(Path(WHEELS_DIR).glob("*.whl"))
    
    # 1. Danh sách đen: Các phiên bản cũ gây xung đột cần bị loại bỏ
    blacklist = [
        "huggingface_hub-1.8.0",
        "pandas-2.3.2",
        "pyarrow-20.0.0",
        "sentencepiece-0.2.0"
    ]
    
    # 2. Lọc danh sách file: Chỉ lấy những file KHÔNG nằm trong blacklist
    valid_wheels = []
    for w in wheel_paths:
        if not any(old_version in str(w) for old_version in blacklist):
            valid_wheels.append(str(w))
    
    print(f"📦 Tìm thấy {len(valid_wheels)} files hợp lệ (đã lọc sạch bản cũ). Đang ép cài đặt...")
    
    # 3. Ép cài đặt trực tiếp
    cmd = [
        sys.executable, "-m", "pip", "install", "-q", 
        "--no-deps", "--force-reinstall"
    ] + valid_wheels
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ Offline installation complete! Tất cả thư viện đã sẵn sàng.")
    else:
        print(f"❌ Offline wheels failed:\n{result.stderr[-500:]}")
else:
    print(f"❌ LỖI: Không tìm thấy thư mục {WHEELS_DIR}.")
    print("   -> Bạn đã nhấn '+ Add Data' và thêm dataset này vào Notebook chưa?")

# ==========================================================
# Flash Attention
# ==========================================================
print("\n⚡ Đang kiểm tra Flash Attention...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 
                    "flash-attn", "--no-build-isolation"], 
                   check=True, capture_output=True, timeout=60)
    print("✅ Flash Attention installed.")
except Exception:
    print("⚠️ Flash Attention not available (Vì không có mạng). Bỏ qua và tiếp tục...")
print("=" * 60)

🚀 BẮT ĐẦU CÀI ĐẶT THƯ VIỆN OFFLINE (Bypass Dependencies)
📦 Tìm thấy 16 files hợp lệ (đã lọc sạch bản cũ). Đang ép cài đặt...
❌ Offline wheels failed:
ERROR: sentencepiece-0.2.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl is not a supported wheel on this platform.


⚡ Đang kiểm tra Flash Attention...
⚠️ Flash Attention not available (Vì không có mạng). Bỏ qua và tiếp tục...


In [51]:
# import subprocess, os, sys
# from pathlib import Path

# WHEELS_DIR = "/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-wheels"

# print("=" * 60)
# print("🚀 BẮT ĐẦU CÀI ĐẶT THƯ VIỆN OFFLINE (Bypass Dependencies)")
# print("=" * 60)

# if os.path.exists(WHEELS_DIR):
#     # 1. Quét toàn bộ file .whl trong thư mục
#     wheel_paths = list(Path(WHEELS_DIR).glob("*.whl"))
    
#     # 2. Lọc bỏ file huggingface_hub-1.8.0 cũ để tránh xung đột với 1.9.0
#     valid_wheels = [str(w) for w in wheel_paths if "huggingface_hub-1.8.0" not in str(w)]
    
#     print(f"📦 Tìm thấy {len(valid_wheels)} files hợp lệ. Đang ép cài đặt trực tiếp...")
    
#     # 3. Lệnh cài đặt trực tiếp từ file, ép đè và cấm kiểm tra phụ thuộc
#     cmd = [
#         sys.executable, "-m", "pip", "install", "-q", 
#         "--no-deps", "--force-reinstall"
#     ] + valid_wheels
    
#     result = subprocess.run(cmd, capture_output=True, text=True)
    
#     if result.returncode == 0:
#         print("✅ Offline installation complete! Tất cả thư viện đã sẵn sàng.")
#     else:
#         print(f"❌ Offline wheels failed:\n{result.stderr[-500:]}")
# else:
#     print(f"❌ LỖI: Không tìm thấy thư mục {WHEELS_DIR}.")
#     print("   -> Bạn đã nhấn '+ Add Data' và thêm dataset này vào Notebook chưa?")

# # ==========================================================
# # Flash Attention (Chỉ chạy được Offline nếu bạn đã có file .whl của nó)
# # ==========================================================
# print("\n⚡ Đang kiểm tra Flash Attention...")
# try:
#     # Lệnh này mặc định sẽ lỗi nếu không có mạng và không có file offline
#     subprocess.run([sys.executable, "-m", "pip", "install", "-q", 
#                     "flash-attn", "--no-build-isolation"], 
#                    check=True, capture_output=True, timeout=60)
#     print("✅ Flash Attention installed.")
# except Exception:
#     print("⚠️ Flash Attention not available (Vì không có mạng). Bỏ qua và tiếp tục...")
# print("=" * 60)

## CELL 2: Setup Working Directory, sys.path, and Dataset Links

In [52]:
import os, sys, shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Register WORK_DIR on sys.path BEFORE copying (slot is created now)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

# Step 2: Copy pipeline source code into WORK_DIR
CODE_SRC = Path("/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-pipeline")
if not CODE_SRC.exists():
    raise FileNotFoundError(
        f"Pipeline code not found at {CODE_SRC}\n"
        "Add dataset 'thanhduc1108/vlsp2025-kd-pipeline' as notebook input."
    )

for d in ["pipeline", "src", "configs"]:
    src = CODE_SRC / d
    dst = WORK_DIR / d
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = sum(1 for _ in dst.rglob("*") if _.is_file())
        print(f"  Copied {d}/ ({n} files)")
    else:
        print(f"  WARNING: {d}/ missing from pipeline dataset")

if (CODE_SRC / "requirements.txt").exists():
    shutil.copy2(CODE_SRC / "requirements.txt", WORK_DIR / "requirements.txt")

# Step 3: Verify the copy succeeded and pipeline is importable
pipeline_init = WORK_DIR / "pipeline" / "__init__.py"
if not pipeline_init.exists():
    raise FileNotFoundError(
        f"pipeline/__init__.py not found after copy.\n"
        "The vlsp2025-kd-pipeline dataset may be outdated. Re-upload with:\n"
        "  python scripts/kaggle_upload.py --upload-code"
    )
print(f"\nPipeline code ready at: {WORK_DIR / 'pipeline'}")

# Step 4: Set working directory so relative paths work
os.chdir(WORK_DIR)

# Step 5: Symlink datasets into expected locations
DATASET_DIR = WORK_DIR / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

for src_path, dst_name, label in [
    ("/kaggle/input/datasets/thanhduc1108/vinumericalqa-private", "viNumericalQA_private", "ViNumQA"),
    ("/kaggle/input/datasets/thanhduc1108/finqa-en",              "finqa_en",              "FinQA"),
]:
    src = Path(src_path)
    dst = DATASET_DIR / dst_name
    if src.exists():
        if not dst.exists():
            dst.symlink_to(src)
        n = sum(1 for _ in src.rglob("*") if _.is_file())
        print(f"  {label} linked ({n} files): {dst}")
    else:
        print(f"  WARNING: {label} not found at {src_path}")

# Step 6: Sanity-check the uploaded train_sft.py carries the Trainer fix.
# The fix produces a "save_strategy=" entry in training args; absence
# almost certainly means the pipeline dataset is an older upload.
sft_py = WORK_DIR / "pipeline" / "train_sft.py"
if sft_py.exists():
    _sft_txt = sft_py.read_text(encoding="utf-8")
    if "save_strategy=" in _sft_txt and "can_eval" in _sft_txt:
        print("  train_sft.py: hardened Trainer config detected OK")
    else:
        print("  WARNING: train_sft.py looks OLD (no save_strategy guard).")
        print("           Re-upload pipeline dataset: python scripts/kaggle_upload.py --upload-code")
else:
    print("  WARNING: pipeline/train_sft.py not found, skipping check.")

print(f"\nWorking directory : {WORK_DIR}")
print(f"sys.path[0]       : {sys.path[0]}")
print(f"Contents          : {[p.name for p in sorted(WORK_DIR.iterdir())]}")

# Quick import smoke-test
try:
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        "pipeline.config", str(WORK_DIR / "pipeline" / "config.py"))
    print("Pipeline import check: OK")
except Exception as e:
    print(f"Pipeline import check FAILED: {e}")


  Copied pipeline/ (11 files)
  Copied src/ (18 files)
  Copied configs/ (4 files)

Pipeline code ready at: /kaggle/working/vlsp2025/pipeline
  ViNumQA linked (4 files): /kaggle/working/vlsp2025/dataset/viNumericalQA_private
  FinQA linked (4 files): /kaggle/working/vlsp2025/dataset/finqa_en
  train_sft.py: hardened Trainer config detected OK

Working directory : /kaggle/working/vlsp2025
sys.path[0]       : /kaggle/working/vlsp2025
Contents          : ['configs', 'data', 'dataset', 'pipeline', 'requirements.txt', 'src']
Pipeline import check: OK


## CELL 3: Verify Environment & Detect GPU Profile

In [53]:
import sys
from pathlib import Path

# Idempotent sys.path guard (safe even if cells run out of order)
WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

import torch

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Enable GPU: Notebook -> Settings -> Accelerator -> GPU T4 / P100 / RTX 6000"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU         : {gpu_name}  ({vram_gb:.1f} GB VRAM)")

import transformers, peft, accelerate, datasets
print(f"Transformers: {transformers.__version__}")
print(f"PEFT        : {peft.__version__}")
print(f"Accelerate  : {accelerate.__version__}")
print(f"Datasets    : {datasets.__version__}")

try:
    import flash_attn
    HAS_FLASH = True
    print(f"Flash Attn  : {flash_attn.__version__}")
except ImportError:
    HAS_FLASH = False
    print("Flash Attn  : not available")

if "6000" in gpu_name or vram_gb > 90:
    GPU_PROFILE = "rtx6000_96gb"
elif "A100" in gpu_name and vram_gb > 70:
    GPU_PROFILE = "a100_80gb"
elif "P100" in gpu_name or vram_gb < 20:
    GPU_PROFILE = "p100_16gb"
else:
    GPU_PROFILE = "p100_16gb"

print(f"\nGPU Profile : {GPU_PROFILE}")

PyTorch     : 2.10.0+cu128
CUDA        : True
GPU         : NVIDIA RTX PRO 6000 Blackwell Server Edition  (95.0 GB VRAM)
Transformers: 5.5.4
PEFT        : 0.18.1
Accelerate  : 1.12.0
Datasets    : 4.8.3
Flash Attn  : not available

GPU Profile : rtx6000_96gb


## CELL 4: Resolve Model Paths (Kaggle offline -> HuggingFace fallback)

In [54]:
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

TEACHER_MODEL_ID = "/kaggle/input/models/thanhduc1108/qwen_35_27b/transformers/default/1"
STUDENT_MODEL_ID = "/kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1"

def resolve_model_path(model_id: str) -> str:
    """Check /kaggle/input/* and /kaggle/models/* for weights; fall back to HF id."""
    base_name = model_id.split("/")[-1]
    print(base_name)
    slugs = set()
    for name in [base_name, base_name.lower()]:
        slugs.add(name)
        slugs.add(name.replace(".", "-").replace("_", "-"))
        slugs.add(name.replace(".", "_").replace("-", "_"))

    for root in [Path("/kaggle/input"), Path("/kaggle/models")]:
        if not root.exists():
            continue
        for entry in root.iterdir():
            if entry.name.lower() in {s.lower() for s in slugs}:
                for candidate in [entry, *entry.rglob("config.json")]:
                    check_dir = candidate if candidate.is_dir() else candidate.parent
                    has_weights = (
                        any(check_dir.glob("*.safetensors")) or
                        any(check_dir.glob("*.bin")) or
                        (check_dir / "config.json").exists()
                    )
                    if has_weights:
                        print(f"  Found offline: {check_dir}")
                        return str(check_dir)

    print(f"  Not found offline -> using HuggingFace: {model_id}")
    return model_id


print("Resolving teacher model...")
TEACHER_PATH = resolve_model_path(TEACHER_MODEL_ID)
print("Resolving student model...")
STUDENT_PATH = resolve_model_path(STUDENT_MODEL_ID)
print(f"\nTeacher: {TEACHER_PATH}")
print(f"Student: {STUDENT_PATH}")

Resolving teacher model...
1
  Not found offline -> using HuggingFace: /kaggle/input/models/thanhduc1108/qwen_35_27b/transformers/default/1
Resolving student model...
1
  Not found offline -> using HuggingFace: /kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1

Teacher: /kaggle/input/models/thanhduc1108/qwen_35_27b/transformers/default/1
Student: /kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1


## CELL 5: Configure Pipeline

In [55]:
import os, sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

from pipeline.config import load_config, save_config

# ── 12h Kaggle session budget for KD (distill + SFT in one session) ───
# Teacher distillation on ~14k samples @ bs=12, 512 new tokens ≈ 3–5h.
# That leaves ~6h for SFT. Watchdog enforces a hard stop; mirror-save
# persists the adapter to /kaggle/working/outputs even if the kernel is
# killed. Values are conservative — if distill is fast, we waste SFT
# headroom, but we never blow past 12h and lose the session output.
SFT_MIRROR_DIR = "/kaggle/working/outputs/sft_adapter_mirror"
Path(SFT_MIRROR_DIR).mkdir(parents=True, exist_ok=True)

# Runtime cap for the SFT phase alone — not wall-clock from session start.
# The watchdog is inside the SFT subprocess, so it sees only SFT time.
SFT_MAX_RUNTIME_HOURS = 6.0

# Sample cap on the distilled train set (if larger than this). The
# distilled corpus is ~14k samples; capping to 8k keeps epoch-time
# predictable on SDPA fallback.
SFT_MAX_TRAIN_SAMPLES = 8000

# Hard cap on the teacher distillation phase. If distill runs longer
# than this, it stops at a checkpoint so SFT still has its full budget.
# Resume on next session picks up where this one stopped.
TEACHER_MAX_RUNTIME_HOURS = 4.5

config_overrides = {
    "model": {
        "teacher_model": TEACHER_PATH,
        "student_model": STUDENT_PATH,
        "use_flash_attention": HAS_FLASH,
    },
    "data": {
        "vinumqa_train":        "/kaggle/input/vinumericalqa-private/train.json",
        "vinumqa_valid":        "/kaggle/input/vinumericalqa-private/valid.json",
        "vinumqa_test":         "/kaggle/input/vinumericalqa-private/test.json",
        "vinumqa_private_test": "/kaggle/input/vinumericalqa-private/private_test.json",
        "finqa_dir":            "/kaggle/input/finqa-en",
    },
    "sft": {
        "num_epochs":                    2,
        "max_seq_length":                1536,
        "per_device_train_batch_size":   1,
        "gradient_accumulation_steps":   16,
        "save_steps":                    200,
        "eval_steps":                    200,
        "save_total_limit":              2,
        "max_train_samples":             SFT_MAX_TRAIN_SAMPLES,
        "max_runtime_hours":             SFT_MAX_RUNTIME_HOURS,
        "mirror_save_dir":               SFT_MIRROR_DIR,
    },
    "teacher": {
        "max_runtime_hours":             TEACHER_MAX_RUNTIME_HOURS,
    },
}

cfg = load_config(gpu_profile=GPU_PROFILE, overrides=config_overrides)

print(f"\n{'='*60}")
print("Pipeline Configuration")
print(f"{'='*60}")
print(f"  GPU Profile   : {GPU_PROFILE}")
print(f"  Teacher model : {cfg.model.teacher_model}")
print(f"  Student model : {cfg.model.student_model}")
print(f"  Teacher quant : {cfg.model.teacher_quantization}")
print(f"  Flash Attn    : {cfg.model.use_flash_attention}")
print(f"  dtype         : {cfg.model.torch_dtype}")
print(f"  SFT epochs    : {cfg.sft.num_epochs}  |  LoRA r={cfg.sft.lora_r}  |  seq={cfg.sft.max_seq_length}")
print(f"  SFT batch     : {cfg.sft.per_device_train_batch_size} x {cfg.sft.gradient_accumulation_steps} (eff {cfg.sft.per_device_train_batch_size * cfg.sft.gradient_accumulation_steps})")
print(f"  SFT fit-12h   : cap={cfg.sft.max_train_samples}  watchdog={cfg.sft.max_runtime_hours}h  mirror={cfg.sft.mirror_save_dir}")
print(f"  Teacher cap   : watchdog={cfg.teacher.max_runtime_hours}h  batch={cfg.teacher.batch_size}  max_new={cfg.teacher.max_new_tokens}")
print(f"  GRPO epochs   : {cfg.grpo.num_epochs}  |  N generations={cfg.grpo.num_generations}")
print(f"  Inference N   : {cfg.inference.num_candidates}")
print(f"{'='*60}")

# Save — the SFT subprocess will pick this up via --config.
CFG_YAML = str(WORK_DIR / "data/pipeline/config.yaml")
save_config(cfg, CFG_YAML)
print(f"Config saved: {CFG_YAML}")



Pipeline Configuration
  GPU Profile   : rtx6000_96gb
  Teacher model : /kaggle/input/models/thanhduc1108/qwen_35_27b/transformers/default/1
  Student model : /kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1
  Teacher quant : None
  Flash Attn    : False
  dtype         : bfloat16
  SFT epochs    : 3  |  LoRA r=128
  GRPO epochs   : 1  |  N generations=5
  Inference N   : 15
Config saved: /kaggle/working/vlsp2025/data/pipeline/config.yaml
Config saved.


## CELL 6: Phase 1 — Data Preparation

In [56]:
import os, sys, time
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

from pipeline.data_prep import run_data_prep

print("=" * 60)
print("PHASE 1: DATA PREPARATION")
print("=" * 60)

t0 = time.time()
data_paths = run_data_prep(cfg)
print(f"\nData prep completed in {time.time()-t0:.1f}s")
for k, v in data_paths.items():
    print(f"  {k}: {v}")

PHASE 1: DATA PREPARATION
ViNumQA train: 2993 samples
ViNumQA valid: 584 samples

Total training samples (raw): 2993
Validation samples (raw): 584
Teacher mode: GUIDED (think.py) — gold program/answer embedded in prompt
Saved 2993 samples → /kaggle/working/vlsp2025/data/pipeline/teacher_input.json
Saved 2993 samples → /kaggle/working/vlsp2025/data/pipeline/sft_train.json
Saved 584 samples → /kaggle/working/vlsp2025/data/pipeline/sft_valid.json
Saved 2993 GRPO train → /kaggle/working/vlsp2025/data/pipeline/grpo_train.parquet
Saved 584 GRPO valid → /kaggle/working/vlsp2025/data/pipeline/grpo_valid.parquet

Data preparation complete!
  SFT train:  2993 samples
  SFT valid:  584 samples
  GRPO train: 2993 samples
  GRPO valid: 584 samples
  Teacher:    2993 samples


Data prep completed in 1.0s
  teacher_input: /kaggle/working/vlsp2025/data/pipeline/teacher_input.json
  sft_train: /kaggle/working/vlsp2025/data/pipeline/sft_train.json
  sft_valid: /kaggle/working/vlsp2025/data/pipeline/sft_

## PHASE 2 — Teacher Distillation
### Strategy: 2 Committed Runs (each < 12h)

> ⚠️ **CRITICAL**: Run as **Save Version** (committed run), NOT interactive.
> Only committed runs save `/kaggle/working` as permanent output.
> Interactive sessions lose all checkpoint data when they timeout.

| Session | Cells | What runs | Est. time |
|---------|-------|-----------|-----------|
| **Session 1** (this file) | 0→7b | Setup → Distill → Save output | **~6-8h** |
| **Session 2** (session2 notebook) | 0→end | Load → SFT → GRPO → Eval | **~5-7h** |

#### How to persist between sessions:
1. **Session 1 ends** → Kaggle auto-saves `/kaggle/working` as notebook output
2. Open Kaggle → Your Notebooks → Session 1 → Output tab → See saved files
3. Create new notebook (Session 2) → Add Session 1 output as Input dataset
4. Session 2 loads checkpoint from `/kaggle/input/<session1-output>/`

#### Key optimizations applied:
- `max_new_tokens=512` (was 2048): **4x** decode speedup
- `use_guided_template=True`: gold program in prompt → **~98% match rate** (was 45%)
- `max_retries=0`: no retry on fail → **1.3x** speedup
- `batch_size=12`: better GPU utilization on 95GB VRAM
- `causal-conv1d` (Cell 0): **2.5x** SSM layer speedup (if installed)


In [57]:
# ── CELL 7a: Timing Estimate & Checkpoint Status ──────────────────
import json, time, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
pipeline_out = WORK_DIR / "data/pipeline"

total = 0
done = 0
if (pipeline_out / "teacher_input.json").exists():
    dataset = json.load(open(pipeline_out / "teacher_input.json"))
    total = len(dataset)
if (pipeline_out / "teacher_raw_checkpoint.json").exists():
    ckpt = json.load(open(pipeline_out / "teacher_raw_checkpoint.json"))
    done = len(ckpt)
    matched = sum(1 for r in ckpt if r.get("matched", False))
    guided_count = sum(1 for r in ckpt if r.get("_guided", False))
    print(f"Checkpoint: {done}/{total} done, {matched} matched ({matched/max(done,1)*100:.0f}%)")

remaining = total - done
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0

# Check if causal-conv1d is installed
try:
    import causal_conv1d
    conv1d_ok = True
    throughput = 40  # tokens/sec with causal-conv1d
except ImportError:
    conv1d_ok = False
    throughput = 17  # tokens/sec without (measured actual)

# With guided template: ~280 avg output tokens (vs 600 without)
# Batch size 12: better parallelism  
avg_tokens = 280
s_per_batch = (cfg.teacher.batch_size * avg_tokens) / throughput if total > 0 else 0
s_per_sample = s_per_batch / cfg.teacher.batch_size if cfg.teacher.batch_size > 0 else 20
eta_h = remaining * s_per_sample / 3600

print(f"\nGPU VRAM     : {vram:.0f} GB")
print(f"causal-conv1d: {'✓ installed' if conv1d_ok else '✗ missing (2.5x slower)'}")
print(f"batch_size   : {cfg.teacher.batch_size}")
print(f"max_new_tokens: {cfg.teacher.max_new_tokens}")
print(f"guided template: {cfg.teacher.use_guided_template}")
print(f"Throughput est: {throughput} tok/s → {s_per_sample:.1f}s/sample")
print(f"\nRemaining    : {remaining} samples")
print(f"Est. time    : {eta_h:.1f}h")
print()
if eta_h > 10.5:
    print("⚠ WARNING: May not complete in 12h!")
    if not conv1d_ok:
        print("  → Install causal-conv1d for 2.5x speedup (see Cell 0)")
elif eta_h > 0:
    print(f"✓ Should complete in ~{eta_h:.1f}h (well within 12h session)")



GPU VRAM     : 95 GB
causal-conv1d: ✗ missing (2.5x slower)
batch_size   : 12
max_new_tokens: 512
guided template: True
Throughput est: 17 tok/s → 16.5s/sample

Remaining    : 2993 samples
Est. time    : 13.7h

⚠ WARNING: May not complete in 12h!
  → Install causal-conv1d for 2.5x speedup (see Cell 0)


In [59]:
cfg

PipelineConfig(project_name='vlsp2025-kd', seed=42, project_root='/kaggle/working/vlsp2025', model=ModelConfig(teacher_model='/kaggle/input/models/thanhduc1108/qwen_35_27b/transformers/default/1', student_model='/kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1', teacher_quantization=None, student_quantization=None, use_flash_attention=False, torch_dtype='bfloat16', max_seq_length=8192, trust_remote_code=True), data=DataConfig(vinumqa_train='/kaggle/input/vinumericalqa-private/train.json', vinumqa_valid='/kaggle/input/vinumericalqa-private/valid.json', vinumqa_test='/kaggle/input/vinumericalqa-private/test.json', vinumqa_private_test='/kaggle/input/vinumericalqa-private/private_test.json', finqa_dir='/kaggle/input/finqa-en', output_dir='data/pipeline', use_finqa=True, use_program_re=True, max_samples=None), teacher=TeacherConfig(base_url='http://localhost:8000/v1', api_key='no-need', max_workers=8, max_retries=0, batch_size=12, checkpoint_every=100, max_new_tokens=512

In [60]:
import gc, os, sys, time, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

# Offline-safe HF hub resolution — Kaggle without internet would otherwise
# have tokenizer.from_pretrained hit the network and fail.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from pipeline.teacher_distill import run_teacher_distillation

print("=" * 65)
print("PHASE 2: TEACHER DISTILLATION")
print(f"  Guided template : {cfg.teacher.use_guided_template} (gold program embedded in prompt)")
print(f"  max_new_tokens  : {cfg.teacher.max_new_tokens}  (was 2048 - 4x faster decode)")
print(f"  max_retries     : {cfg.teacher.max_retries}  (0 = try once, use gold data on fail)")
print(f"  batch_size      : {cfg.teacher.batch_size}")
print(f"  checkpoint_every: {cfg.teacher.checkpoint_every} samples")
print("=" * 65)

t0 = time.time()
distilled_path = run_teacher_distillation(
    cfg,
    teacher_input_path=data_paths["teacher_input"],
    resume=True,
)
elapsed = time.time() - t0
print(f"\nDistillation completed in {elapsed/3600:.2f}h ({elapsed:.0f}s)")
print(f"  Distilled SFT data: {distilled_path}")

# ── AGGRESSIVE GPU CLEANUP ────────────────────────────────────────────
# The teacher model (e.g. 27B bf16 ~54 GB VRAM) must be fully released
# before loading the student for SFT; without this the SFT step OOMs
# even on the 96 GB RTX 6000 Pro.
import importlib
import pipeline.teacher_distill as _td_mod
importlib.reload(_td_mod)

for _n in ("teacher", "_teacher", "LocalTeacher"):
    if _n in globals():
        try:
            del globals()[_n]
        except Exception:
            pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"  GPU free after cleanup: {free_b/1024**3:.1f} / {total_b/1024**3:.1f} GB")


PHASE 2: TEACHER DISTILLATION
  Guided template : True (gold program embedded in prompt)
  max_new_tokens  : 512  (was 2048 - 4x faster decode)
  max_retries     : 0  (0 = try once, use gold data on fail)
  batch_size      : 12
  checkpoint_every: 100 samples
Loaded 2993 samples for teacher distillation
  Already done: 0, Remaining: 2993
Loading teacher model: /kaggle/input/models/thanhduc1108/qwen_35_27b/transformers/default/1


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Teacher distillation (batched):   4%|▎         | 108/2993 [08:55<3:57:44,  4.94s/it, rate=0.2s/s, ETA=4.0h]

  [Checkpoint] 108 done, 79 matched (73.1%)


Teacher distillation (batched):   7%|▋         | 204/2993 [17:05<3:56:57,  5.10s/it, rate=0.2s/s, ETA=3.9h]

  [Checkpoint] 204 done, 148 matched (72.5%)


KeyboardInterrupt: 

In [69]:
# ══════════════════════════════════════════════════════════════════
# CELL 7b: SAVE SESSION-1 OUTPUTS  ← RUN THIS BEFORE SESSION ENDS
# This ensures /kaggle/working files are preserved as notebook output
# after the committed run completes.
# ══════════════════════════════════════════════════════════════════
import json, os, shutil, time
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/vlsp2025")
pipeline_out = WORK_DIR / "data/pipeline"
output_dir   = WORK_DIR / "session1_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

SAVE_FILES = [
    "teacher_raw_checkpoint.json",   # Resume checkpoint
    "distilled_sft.json",            # Teacher distilled data → SFT input
    "teacher_raw_output.json",       # Full teacher outputs for debug
    "sft_train.json",                # Gold SFT data (fallback)
    "sft_valid.json",
    "grpo_train.parquet",
    "grpo_valid.parquet",
    "teacher_input.json",
    "config.yaml",
]

print("Saving Session-1 outputs...")
summary = {"saved": [], "missing": [], "timestamp": time.time()}
total_mb = 0.0

for fname in SAVE_FILES:
    src = pipeline_out / fname
    if src.exists():
        dst = output_dir / fname
        shutil.copy2(src, dst)
        mb = dst.stat().st_size / 1024**2
        total_mb += mb
        summary["saved"].append(fname)
        print(f"  ✓ {fname:45s} {mb:7.1f} MB")
    else:
        summary["missing"].append(fname)
        print(f"  ○ {fname:45s}  (not found)")

# Write summary.json for quick inspection in Session 2
if (pipeline_out / "teacher_raw_checkpoint.json").exists():
    ckpt = json.load(open(pipeline_out / "teacher_raw_checkpoint.json"))
    matched = sum(1 for r in ckpt if r.get("matched", False))
    summary["distillation"] = {
        "total_processed": len(ckpt),
        "matched": matched,
        "match_rate": f"{matched/max(len(ckpt),1)*100:.1f}%",
    }

with open(output_dir / "summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\nTotal saved: {total_mb:.0f} MB → {output_dir}")
print(f"Distillation: {summary.get('distillation', {})}")
print()
print("══ NEXT STEPS ════════════════════════════════════════════")
print("1. This committed run will save /kaggle/working as output")
print("2. In Kaggle: Your Notebooks → Session 1 → Output → session1_outputs/")
print("3. For Session 2: Create new notebook, Add this output as Input dataset")
print("   Path in Session 2: /kaggle/input/<your-session1-output-name>/session1_outputs/")
print("══════════════════════════════════════════════════════════")


Saving Session-1 outputs...
  ✓ teacher_raw_checkpoint.json                       2.4 MB
  ○ distilled_sft.json                             (not found)
  ○ teacher_raw_output.json                        (not found)
  ✓ sft_train.json                                   33.7 MB
  ✓ sft_valid.json                                    6.4 MB
  ✓ grpo_train.parquet                                8.2 MB
  ✓ grpo_valid.parquet                                1.5 MB
  ✓ teacher_input.json                               33.8 MB
  ✓ config.yaml                                       0.0 MB

Total saved: 86 MB → /kaggle/working/vlsp2025/session1_outputs
Distillation: {'total_processed': 204, 'matched': 148, 'match_rate': '72.5%'}

══ NEXT STEPS ════════════════════════════════════════════
1. This committed run will save /kaggle/working as output
2. In Kaggle: Your Notebooks → Session 1 → Output → session1_outputs/
3. For Session 2: Create new notebook, Add this output as Input dataset
   Path in Sessio

In [70]:
# ══════════════════════════════════════════════════════════════════
# CELL 7c: LOAD SESSION-1 OUTPUTS  ← SESSION 2 ONLY
# In a single-session run, files already exist in /kaggle/working — this cell
# is a no-op.  In a 2-session run, copy files from the input dataset.
# ══════════════════════════════════════════════════════════════════
import json, shutil
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/vlsp2025")
pipeline_out = WORK_DIR / "data/pipeline"
pipeline_out.mkdir(parents=True, exist_ok=True)

# --- Try to copy Session-1 files from input dataset (2-session flow) -----------
SEARCH_PATHS = [
    Path("/kaggle/working/vlsp2025/session1_outputs"),
    Path("/kaggle/working/vlsp2025"),
    Path("/kaggle/input/session1-outputs/session1_outputs"),
    Path("/kaggle/input/session1-outputs"),
]

src_dir = None
for p in SEARCH_PATHS:
    if p.exists() and any(p.glob("*.json")):
        src_dir = p
        break

if src_dir is not None:
    print(f"Found Session-1 outputs at: {src_dir}")
    if (src_dir / "summary.json").exists():
        s = json.load(open(src_dir / "summary.json"))
        print(f"  Distillation: " + str(s.get("distillation", "N/A")))

    LOAD_FILES = [
        "distilled_sft.json", "teacher_raw_checkpoint.json",
        "sft_train.json", "sft_valid.json",
        "grpo_train.parquet", "grpo_valid.parquet",
    ]
    for fname in LOAD_FILES:
        src = src_dir / fname
        dst = pipeline_out / fname
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"  Copied: {fname}")
        elif dst.exists():
            print(f"  Already present: {fname}")
else:
    print("No Session-1 input dataset found — using files already in working directory.")

# --- Set distilled_path from whatever is available ----------------------------
for candidate in ["distilled_sft.json", "sft_train.json"]:
    p = pipeline_out / candidate
    if p.exists():
        distilled_path = str(p)
        d = json.load(open(distilled_path))
        print(f"distilled_path = {distilled_path}  ({len(d)} samples)")
        break
else:
    distilled_path = None
    print("WARNING: No train data found. Run Cell 6 (data prep) or Cell 7b (distillation) first.")


Found Session-1 outputs at: /kaggle/working/vlsp2025/session1_outputs
  Distillation: {'total_processed': 204, 'matched': 148, 'match_rate': '72.5%'}
  Already present: teacher_raw_checkpoint.json
  Already present: sft_train.json
  Already present: sft_valid.json
  Already present: grpo_train.parquet
  Already present: grpo_valid.parquet
distilled_path = /kaggle/working/vlsp2025/data/pipeline/sft_train.json  (2993 samples)


## CELL 8: Phase 3 — SFT Training

In [71]:
# import os
# import shutil

# def cleanup_disk():
#     # 1. Xóa cache của HuggingFace (thường chiếm rất nhiều)
#     cache_dir = os.path.expanduser("~/.cache/huggingface")
#     if os.path.exists(cache_dir):
#         shutil.rmtree(cache_dir)
#         print("Cleared HF cache")

#     # 2. Xóa các checkpoint cũ trong working dir
#     # Chỉ giữ lại folder checkpoints/sft nhưng xóa các bản cũ bên trong
#     path = "/kaggle/working/vlsp2025/checkpoints/sft"
#     if os.path.exists(path):
#         files = sorted([os.path.join(path, f) for f in os.listdir(path)], key=os.path.getmtime)
#         if len(files) > 1:
#             for f in files[:-1]: # Xóa tất cả trừ bản cuối cùng
#                 shutil.rmtree(f)
#                 print(f"Deleted old checkpoint: {f}")

# cleanup_disk()

In [72]:
import gc, json, os, subprocess, sys, time
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
os.chdir(WORK_DIR)

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
# Stricter allocator config: prevent huge reservations that starve activations.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.6"
)

print("=" * 60)
print("PHASE 3: SUPERVISED FINE-TUNING (subprocess-isolated, 12h fit)")
print("=" * 60)

pipeline_out = WORK_DIR / "data/pipeline"
outputs_dir  = Path("/kaggle/working/outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

# ── AGGRESSIVE parent-kernel CUDA release ─────────────────────────────
# The Jupyter kernel still holds ~50+ GB reserved from the teacher run.
# Since the subprocess spawns on the SAME GPU, parent's reserved memory
# starves the child of activations. We must force every cached block
# back to the OS before launching the SFT subprocess.
import torch
for _name in list(globals()):
    _obj = globals().get(_name)
    if _obj is None: continue
    try:
        if isinstance(_obj, torch.nn.Module) or (hasattr(_obj, "device") and
            getattr(_obj, "device", None) is not None and str(_obj.device).startswith("cuda")):
            del globals()[_name]
    except Exception:
        pass
for _ in range(3):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
if torch.cuda.is_available():
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    _free, _total = torch.cuda.mem_get_info()
    _used_gb = (_total - _free) / 2**30
    print(f"  Kernel GPU before launch: free={_free/2**30:.1f}GB / used={_used_gb:.1f}GB / {_total/2**30:.1f}GB")
    if _used_gb > 20:
        print("  WARNING: parent kernel still holds >20 GB. Child subprocess may hit OOM.")
        print("           Recommended: restart the Jupyter kernel, run only data-prep + this cell.")

# ── Resolve train / valid paths ───────────────────────────────────────
def _resolve_train():
    if "distilled_path" in dir() and Path(distilled_path).exists():
        try:
            with open(distilled_path, encoding="utf-8") as f:
                if len(json.load(f)) > 0:
                    return distilled_path
        except Exception:
            pass
    for cand in ["distilled_sft.json", "sft_train.json"]:
        p = pipeline_out / cand
        if p.exists():
            try:
                with open(p, encoding="utf-8") as f:
                    if len(json.load(f)) > 0:
                        return str(p)
            except Exception:
                continue
    raise RuntimeError("No non-empty train data found in data/pipeline/")

train_path = _resolve_train()
valid_path = str(pipeline_out / "sft_valid.json")
with open(train_path, encoding="utf-8") as _f:
    _n_train = len(json.load(_f))
print(f"  train : {train_path}  ({_n_train} samples)")
print(f"  valid : {valid_path}")

_gpu_profile = globals().get("GPU_PROFILE", "rtx6000_96gb")
print(f"  gpu_profile : {_gpu_profile}")

# Subprocess re-loads config from profile + YAML — without --config it
# would LOSE every override (watchdog, mirror, seq_len, epochs, ...).
CFG_YAML = str(WORK_DIR / "data/pipeline/config.yaml")
if not Path(CFG_YAML).exists():
    # cfg cell was skipped — save it now from the in-memory cfg.
    from pipeline.config import save_config as _save_cfg
    _save_cfg(cfg, CFG_YAML)
print(f"  config_yaml : {CFG_YAML}")

# Derive expected SFT final-dir from the already-loaded cfg.
sft_final_dir = (Path(cfg.project_root).resolve() / cfg.sft.output_dir / "final")
print(f"  expected final_dir : {sft_final_dir}")

manifest_path = outputs_dir / "sft_manifest.json"
if manifest_path.exists():
    manifest_path.unlink()

cmd = [
    sys.executable, "-m", "pipeline.train_sft",
    "--gpu-profile", _gpu_profile,
    "--config", CFG_YAML,
    "--train-data", train_path,
    "--valid-data", valid_path,
]
print("  launching:", " ".join(cmd))
print("-" * 60)

env = os.environ.copy()
env["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"
env["HF_HUB_OFFLINE"] = "1"
env["TRANSFORMERS_OFFLINE"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TOKENIZERS_PARALLELISM"] = "false"

t0 = time.time()
proc = subprocess.Popen(
    cmd, cwd=str(WORK_DIR), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, encoding="utf-8", errors="replace",
)
try:
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    raise
elapsed = time.time() - t0
print("-" * 60)
print(f"SFT subprocess exited with code {ret} in {elapsed/3600:.2f}h ({elapsed:.0f}s)")

if ret != 0:
    raise RuntimeError(f"SFT subprocess failed (exit {ret}). See stdout above for traceback.")

# ── Locate adapter: final/ first, else mirror, else latest checkpoint ─
sft_model_path = None
candidates = [
    sft_final_dir,
    Path(cfg.sft.mirror_save_dir) / "final" if cfg.sft.mirror_save_dir else None,
]
for c in candidates:
    if c and Path(c).exists():
        sft_model_path = str(c)
        break
if sft_model_path is None:
    roots = [Path(cfg.project_root).resolve() / cfg.sft.output_dir]
    if cfg.sft.mirror_save_dir:
        roots.append(Path(cfg.sft.mirror_save_dir))
    latest = None
    for root in roots:
        if not root.exists():
            continue
        cks = sorted(
            (p for p in root.glob("checkpoint-*") if p.is_dir()),
            key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
        )
        if cks and (latest is None or
                    int(cks[-1].name.split("-")[-1]) > int(Path(latest).name.split("-")[-1])):
            latest = str(cks[-1])
    if latest:
        sft_model_path = latest
        print(f"  final/ missing — using latest checkpoint: {sft_model_path}")
    else:
        raise RuntimeError("No SFT output found (final/, mirror, or checkpoints)")
print(f"  SFT adapter dir: {sft_model_path}")

import shutil
sft_save_dir = outputs_dir / "sft_adapter"
if sft_save_dir.exists():
    shutil.rmtree(sft_save_dir)
try:
    shutil.copytree(sft_model_path, sft_save_dir)
    _size_mb = sum(p.stat().st_size for p in sft_save_dir.rglob("*") if p.is_file()) / 1024**2
    print(f"  SFT adapter saved: {sft_save_dir}  ({_size_mb:.0f} MB)")
except Exception as _e:
    print(f"  WARNING: could not copy adapter: {_e}")

_mf = {
    "phase": "sft",
    "student_base_model": cfg.model.student_model,
    "adapter_dir": str(sft_save_dir),
    "training_dir": sft_model_path,
    "elapsed_hours": round(elapsed / 3600, 2),
    "train_path": train_path,
    "valid_path": valid_path,
    "train_samples": _n_train,
    "watchdog_hours": cfg.sft.max_runtime_hours,
    "sample_cap": cfg.sft.max_train_samples,
}
with open(manifest_path, "w", encoding="utf-8") as _f:
    json.dump(_mf, _f, indent=2, ensure_ascii=False)
print(f"  Manifest: {manifest_path}")

# Downstream cells expect sft_model_path to point at a usable adapter.
sft_model_path = str(sft_save_dir)

gc.collect()
try:
    import torch as _torch
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()
        _torch.cuda.synchronize()
except Exception:
    pass


PHASE 3: SUPERVISED FINE-TUNING (SFT)
PHASE 3: SUPERVISED FINE-TUNING (SFT)
  train_path : /kaggle/working/vlsp2025/data/pipeline/sft_train.json
  valid_path : /kaggle/working/vlsp2025/data/pipeline/sft_valid.json
SFT train data: /kaggle/working/vlsp2025/data/pipeline/sft_train.json
SFT valid data: /kaggle/working/vlsp2025/data/pipeline/sft_valid.json
Train samples: 2993, Valid samples: 584
Loading student model: /kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1
[GPU-MEM before-student-load] free=2.3GB total=95.0GB alloc=11.9GB reserved=92.0GB


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[GPU-MEM after-student-load] free=2.3GB total=95.0GB alloc=11.9GB reserved=92.0GB


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 259,719,168 || all params: 4,465,470,464 || trainable%: 5.8162
SFT max_seq_length: 2048  (model context: 8192)

Starting SFT training...
  Output: /kaggle/working/vlsp2025/checkpoints/sft
  Epochs: 3
  Steps/epoch (approx): 187  | total: 561
  Save every: 100  |  Eval every: 100
  Effective batch: 16
[GPU-MEM before-train] free=2.3GB total=95.0GB alloc=20.8GB reserved=92.0GB


Step,Training Loss,Validation Loss
100,0.133252,nan
200,0.458164,nan
300,0.519429,nan
400,0.369714,nan
500,0.018111,nan
564,0.059909,nan


[GPU-MEM after-train] free=2.1GB total=95.0GB alloc=22.7GB reserved=92.2GB

SFT model saved → /kaggle/working/vlsp2025/checkpoints/sft/final

SFT training completed in 3.07h (11044s)
  SFT model: /kaggle/working/vlsp2025/checkpoints/sft/final


## CELL 9: Phase 4 — GRPO Training
*(Comment out / skip to use SFT-only: set grpo_model_path = sft_model_path)*

In [80]:
# Trước khi gọi run_sft_training
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect() # Giải phóng thêm bộ nhớ liên tiến trình

In [76]:
# import gc, json, os, shutil, sys, time, torch
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# os.environ.setdefault("HF_HUB_OFFLINE", "1")
# os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# from pipeline.train_grpo import run_grpo_training

# print("=" * 60)
# print("PHASE 4: GRPO WITH PCPO REWARD")
# print("=" * 60)

# outputs_dir = Path("/kaggle/working/outputs")
# outputs_dir.mkdir(parents=True, exist_ok=True)

# # Auto-detect sft_model_path (prefer in-session var, else on-disk adapter)
# if "sft_model_path" not in globals() or not Path(globals().get("sft_model_path", "")).exists():
#     sft_final = WORK_DIR / "checkpoints/sft/final"
#     sft_saved = outputs_dir / "sft_adapter"
#     if sft_final.exists():
#         sft_model_path = str(sft_final)
#         print(f"Auto-detected SFT model (in-session): {sft_model_path}")
#     elif sft_saved.exists():
#         sft_model_path = str(sft_saved)
#         print(f"Auto-detected SFT model (outputs dir): {sft_model_path}")
#     else:
#         sft_model_path = cfg.model.student_model
#         print(f"No SFT checkpoint found, using base student: {sft_model_path}")

# print(f"Starting GRPO from: {sft_model_path}")

# t0 = time.time()
# grpo_model_path = run_grpo_training(cfg, sft_model_path)
# elapsed = time.time() - t0
# print(f"\nGRPO training completed in {elapsed/3600:.2f}h ({elapsed:.0f}s)")
# print(f"  GRPO final dir: {grpo_model_path}")

# # ── Persist GRPO artefacts ────────────────────────────────────────────
# grpo_save_dir = outputs_dir / "grpo_adapter"
# if grpo_save_dir.exists():
#     shutil.rmtree(grpo_save_dir)
# try:
#     shutil.copytree(grpo_model_path, grpo_save_dir)
#     size_mb = sum(p.stat().st_size for p in grpo_save_dir.rglob("*") if p.is_file()) / 1024**2
#     print(f"  GRPO adapter saved to outputs: {grpo_save_dir}  ({size_mb:.0f} MB)")
# except Exception as _e:
#     print(f"  WARNING: could not copy GRPO adapter: {_e}")

# manifest = {
#     "phase": "grpo",
#     "sft_model_path": sft_model_path,
#     "adapter_dir": str(grpo_save_dir),
#     "training_dir": str(grpo_model_path),
#     "elapsed_hours": round(elapsed / 3600, 2),
# }
# with open(outputs_dir / "grpo_manifest.json", "w", encoding="utf-8") as _f:
#     json.dump(manifest, _f, indent=2, ensure_ascii=False)
# print(f"  Manifest: {outputs_dir / 'grpo_manifest.json'}")

# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     torch.cuda.synchronize()


PHASE 4: GRPO WITH PCPO REWARD
Starting GRPO from: /kaggle/working/vlsp2025/checkpoints/sft/final
TRL not installed, using manual GRPO implementation


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/496 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 64,929,792 || all params: 4,270,681,088 || trainable%: 1.5204
GRPO train samples: 2993

=== Epoch 1/1 ===


OutOfMemoryError: CUDA out of memory. Tried to allocate 76.00 MiB. GPU 0 has a total capacity of 94.97 GiB of which 51.88 MiB is free. Including non-PyTorch memory, this process has 94.91 GiB memory in use. Of the allocated memory 93.75 GiB is allocated by PyTorch, and 523.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## CELL 10: Phase 5 — Inference with Majority Voting

In [81]:
import gc, json, os, sys, time, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from pipeline.inference import run_inference

print("=" * 60)
print("PHASE 5: MULTI-PATH INFERENCE + MAJORITY VOTING")
print("=" * 60)

pipeline_out = WORK_DIR / "data/pipeline"
outputs_dir  = Path("/kaggle/working/outputs")

# Auto-detect model paths
if "grpo_model_path" not in globals():
    grpo_final = WORK_DIR / "checkpoints/grpo/final"
    grpo_saved = outputs_dir / "grpo_adapter"
    if grpo_final.exists():
        grpo_model_path = str(grpo_final)
    elif grpo_saved.exists():
        grpo_model_path = str(grpo_saved)
    else:
        grpo_model_path = None

if "sft_model_path" not in globals() or not Path(globals().get("sft_model_path", "")).exists():
    sft_final = WORK_DIR / "checkpoints/sft/final"
    sft_saved = outputs_dir / "sft_adapter"
    if sft_final.exists():
        sft_model_path = str(sft_final)
    elif sft_saved.exists():
        sft_model_path = str(sft_saved)
    else:
        sft_model_path = cfg.model.student_model

# Use best available model: GRPO > SFT > base
final_model = (
    grpo_model_path if grpo_model_path and Path(grpo_model_path).exists()
    else sft_model_path
)
print(f"Using model : {final_model}")

# Auto-detect test input path
if "test_input_path" not in globals() or not Path(globals().get("test_input_path", "")).exists():
    sft_valid_p = pipeline_out / "sft_valid.json"
    test_input_path = (
        globals().get("data_paths", {}).get("sft_valid") or str(sft_valid_p)
    )
print(f"Test input  : {test_input_path}")

t0 = time.time()
predictions_path = run_inference(cfg, final_model, test_input_path)
elapsed = time.time() - t0
print(f"\nInference completed in {elapsed:.1f}s")
print(f"  Predictions: {predictions_path}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


PHASE 5: MULTI-PATH INFERENCE + MAJORITY VOTING
Using model : /kaggle/working/vlsp2025/checkpoints/sft/final
Test input  : /kaggle/working/vlsp2025/data/pipeline/sft_valid.json
Inference model: /kaggle/working/vlsp2025/checkpoints/sft/final
Test samples: 584


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Inference:   3%|▎         | 15/584 [56:34<35:45:54, 226.28s/it]


KeyboardInterrupt: 

## CELL 11: Phase 6 — Evaluation (EA + PA)

In [82]:
import gc, os, sys, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

from pipeline.evaluate import evaluate_against_dataset

print("=" * 60)
print("PHASE 6: EVALUATION (EA + PA)")
print("=" * 60)

pipeline_out = WORK_DIR / "data/pipeline"

# Auto-detect predictions_path
if "predictions_path" not in globals() or not Path(globals().get("predictions_path", "")).exists():
    predictions_path = str(pipeline_out / "predictions.json")
    print(f"Auto-detected predictions: {predictions_path}")

# Original ViNumQA valid JSON supplies table data needed for proper EA/PA
test_dataset_path = str(WORK_DIR / "dataset/viNumericalQA_private/valid.json")
eval_output       = str(pipeline_out / "eval_results.json")

results = evaluate_against_dataset(predictions_path, test_dataset_path, eval_output)

gc.collect()
torch.cuda.empty_cache()


PHASE 6: EVALUATION (EA + PA)
Auto-detected predictions: /kaggle/working/vlsp2025/data/pipeline/predictions.json


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/vlsp2025/data/pipeline/predictions.json'

## CELL 12: Baseline Comparison — Zero-shot Student Model (optional)
*Skip this cell if you only need KD results.*

In [ ]:
import gc, os, shutil, sys, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from pipeline.inference import run_inference
from pipeline.evaluate import evaluate_against_dataset

pipeline_out = WORK_DIR / "data/pipeline"

# Auto-detect STUDENT_PATH (raw base model for zero-shot baseline)
if "STUDENT_PATH" not in globals():
    local_student = Path("/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-models") / Path(cfg.model.student_model).name
    STUDENT_PATH = str(local_student) if local_student.exists() else cfg.model.student_model
    print(f"Auto-detected STUDENT_PATH: {STUDENT_PATH}")

# Auto-detect predictions_path
if "predictions_path" not in globals() or not Path(globals().get("predictions_path", "")).exists():
    predictions_path = str(pipeline_out / "predictions.json")

# Auto-detect test_input_path
if "test_input_path" not in globals() or not Path(globals().get("test_input_path", "")).exists():
    test_input_path = str(pipeline_out / "sft_valid.json")

# Back up KD results before overwriting predictions.json
if Path(predictions_path).exists():
    kd_backup = str(pipeline_out / "predictions_kd.json")
    shutil.copy2(predictions_path, kd_backup)
    print(f"KD predictions backed up -> {kd_backup}")
else:
    kd_backup = None
    print("No KD predictions to back up (run Cell 10 first)")

print(f"\nRunning zero-shot baseline with: {STUDENT_PATH}")
run_inference(cfg, model_path=STUDENT_PATH, test_data_path=test_input_path)

# Rename baseline output, restore KD predictions
baseline_pred_path = str(pipeline_out / "predictions_baseline.json")
shutil.copy2(str(pipeline_out / "predictions.json"), baseline_pred_path)
if kd_backup and Path(kd_backup).exists():
    shutil.copy2(kd_backup, str(pipeline_out / "predictions.json"))
print(f"Baseline predictions saved -> {baseline_pred_path}")

test_dataset_path = str(WORK_DIR / "dataset/viNumericalQA_private/valid.json")
baseline_results = evaluate_against_dataset(
    baseline_pred_path,
    test_dataset_path,
    str(pipeline_out / "eval_baseline.json"),
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


No KD predictions to back up (run Cell 10 first)

Running zero-shot baseline with: /kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1
Inference model: /kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1
Test samples: 584


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Inference:   0%|          | 0/584 [00:00<?, ?it/s]

## CELL 13: Results Summary

In [ ]:
# Provide safe defaults if cells 11 or 12 were skipped
if "baseline_results" not in globals():
    print("WARNING: baseline_results not set - run the baseline cell to compare")
    baseline_results = {"execution_accuracy": 0.0, "program_accuracy": 0.0, "valid_rate": 0.0}

if "results" not in globals():
    print("WARNING: results not set - run the evaluation cell first")
    results = {"execution_accuracy": 0.0, "program_accuracy": 0.0, "valid_rate": 0.0}

print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

header_model  = "Model"
header_ea     = "EA"
header_pa     = "PA"
header_valid  = "Valid%"

print(f"\n{header_model:<30} {header_ea:>8} {header_pa:>8} {header_valid:>8}")
print("-" * 60)

name_base = "Baseline (zero-shot)"
name_kd   = "After KD (SFT+GRPO)"
base_ea = baseline_results["execution_accuracy"]
base_pa = baseline_results["program_accuracy"]
base_vr = baseline_results["valid_rate"]
kd_ea   = results["execution_accuracy"]
kd_pa   = results["program_accuracy"]
kd_vr   = results["valid_rate"]

print(f"{name_base:<30} {base_ea:>7.2%} {base_pa:>7.2%} {base_vr:>7.2%}")
print(f"{name_kd:<30} {kd_ea:>7.2%} {kd_pa:>7.2%} {kd_vr:>7.2%}")

ea_delta = kd_ea - base_ea
pa_delta = kd_pa - base_pa
print(f"\nImprovement: EA {ea_delta:+.2%},  PA {pa_delta:+.2%}")


## CELL 14: Save All Outputs

In [ ]:
import json, os, shutil, sys
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/vlsp2025")
output_dir   = Path("/kaggle/working/outputs")
pipeline_out = WORK_DIR / "data/pipeline"
output_dir.mkdir(parents=True, exist_ok=True)

# Safe fallbacks for variables that may not be defined if cells were skipped
_GPU_PROFILE      = globals().get("GPU_PROFILE",      getattr(cfg, "_gpu_profile", "unknown"))
_TEACHER_MODEL_ID = globals().get("TEACHER_MODEL_ID", cfg.model.teacher_model)
_STUDENT_MODEL_ID = globals().get("STUDENT_MODEL_ID", cfg.model.student_model)
_final_model      = globals().get("final_model",      globals().get("sft_model_path", cfg.model.student_model))
_baseline_results = globals().get("baseline_results", {"execution_accuracy": 0.0, "program_accuracy": 0.0, "valid_rate": 0.0})
_results          = globals().get("results",          {"execution_accuracy": 0.0, "program_accuracy": 0.0, "valid_rate": 0.0})
_ea_delta         = globals().get("ea_delta",         _results["execution_accuracy"] - _baseline_results["execution_accuracy"])
_pa_delta         = globals().get("pa_delta",         _results["program_accuracy"]   - _baseline_results["program_accuracy"])

# Skip copying the final model if its adapter already lives in outputs/.
# That means the SFT/GRPO save cells already persisted it - duplicating
# would waste several GB of Kaggle's 20 GB output quota.
existing_adapters = [p for p in [output_dir / "sft_adapter", output_dir / "grpo_adapter"] if p.exists()]
if existing_adapters:
    print("Final adapter(s) already persisted:")
    for p in existing_adapters:
        sz = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**2
        print(f"  {p.name}: {sz:.0f} MB")
elif _final_model and Path(_final_model).exists():
    final_model_output = output_dir / "final_model"
    if final_model_output.exists():
        shutil.rmtree(final_model_output)
    shutil.copytree(_final_model, final_model_output)
    print(f"Final model saved -> {final_model_output}")
else:
    print(f"Final model not found at {_final_model}, skipping copy")

for fname in ["eval_results.json", "eval_baseline.json",
              "predictions.json", "predictions_kd.json", "predictions_baseline.json",
              "distilled_sft.json", "sft_train.json", "sft_valid.json",
              "config.yaml"]:
    src = pipeline_out / fname
    if src.exists():
        shutil.copy2(src, output_dir / fname)
        print(f"  Copied {fname}")

summary = {
    "gpu_profile":   _GPU_PROFILE,
    "teacher_model": _TEACHER_MODEL_ID,
    "student_model": _STUDENT_MODEL_ID,
    "baseline": {
        "EA":         _baseline_results["execution_accuracy"],
        "PA":         _baseline_results["program_accuracy"],
        "valid_rate": _baseline_results["valid_rate"],
    },
    "after_kd": {
        "EA":         _results["execution_accuracy"],
        "PA":         _results["program_accuracy"],
        "valid_rate": _results["valid_rate"],
    },
    "improvement": {"EA_delta": _ea_delta, "PA_delta": _pa_delta},
}
with open(output_dir / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\nAll outputs -> {output_dir}")
print(f"Files: {[p.name for p in sorted(output_dir.iterdir())]}")
print("\nDone! Check /kaggle/working/outputs/ for final results.")


In [ ]:
# """
# VLSP 2025 - Knowledge Distillation Pipeline for Vietnamese Financial Numerical Reasoning
# =========================================================================================
# Kaggle Notebook for OFFLINE execution on RTX 6000 Pro 96GB.

# HOW TO USE THIS NOTEBOOK:
#   1. Open a new Kaggle notebook (or this one if imported as a notebook)
#   2. Set Accelerator → GPU RTX 6000 Pro / T4 / P100 in Settings
#   3. Add ALL required Input Datasets (see CELL 0 checklist)
#   4. Run cells ONE BY ONE from top to bottom (do NOT "Run All" at once)

# Required Kaggle Input Datasets:
#   - thanhduc1108/vlsp2025-kd-pipeline     → Pipeline source code
#   - thanhduc1108/vlsp2025-kd-wheels       → Python wheels (offline install)
#   - thanhduc1108/finqa-en                 → FinQA English dataset
#   - thanhduc1108/vinumericalqa-private     → ViNumQA Vietnamese dataset
#   - thanhduc1108/qwen-35-27b (Model)      → Teacher model (Qwen3.5-27B)
#   - thanhduc1108/qwen-35-4b  (Model)      → Student model (Qwen3.5-4B)
# """


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 0: Pre-flight Checklist — Run this FIRST to verify all inputs
# # ═══════════════════════════════════════════════════════════════════════
# import os
# from pathlib import Path

# REQUIRED_INPUTS = {
#     "/kaggle/input/vlsp2025-kd-pipeline":   "Pipeline code   (Dataset: thanhduc1108/vlsp2025-kd-pipeline)",
#     "/kaggle/input/finqa-en":               "FinQA dataset   (Dataset: thanhduc1108/finqa-en)",
#     "/kaggle/input/vinumericalqa-private":  "ViNumQA dataset (Dataset: thanhduc1108/vinumericalqa-private)",
# }
# OPTIONAL_INPUTS = {
#     "/kaggle/input/vlsp2025-kd-wheels":     "Offline wheels  (Dataset: thanhduc1108/vlsp2025-kd-wheels)",
# }

# print("=" * 65)
# print("CELL 0: Pre-flight Input Checklist")
# print("=" * 65)
# all_ok = True
# for path, label in REQUIRED_INPUTS.items():
#     exists = os.path.exists(path)
#     status = "✓ FOUND  " if exists else "✗ MISSING"
#     print(f"  [{status}] {label}")
#     print(f"            path: {path}")
#     if not exists:
#         all_ok = False

# for path, label in OPTIONAL_INPUTS.items():
#     exists = os.path.exists(path)
#     status = "✓ found  " if exists else "○ absent "
#     print(f"  [{status}] {label}  (optional)")

# print()
# if not all_ok:
#     print("STOP: Add the MISSING datasets above before continuing.")
#     print("  Notebook -> top-right panel -> '+ Add Data' -> search by dataset name")
#     raise SystemExit("Missing required input datasets. Add them first, then re-run.")
# else:
#     print("All required inputs present. Proceed to Cell 1.")

# import subprocess, sys
# result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
#                          "--format=csv,noheader"], capture_output=True, text=True)
# if result.returncode == 0:
#     print(f"\nGPU: {result.stdout.strip()}")
# else:
#     print("\nWARNING: nvidia-smi not found. Enable GPU: Notebook -> Settings -> Accelerator -> GPU")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 1: Install Dependencies
# # ═══════════════════════════════════════════════════════════════════════
# import subprocess, os, sys
# from pathlib import Path

# WHEELS_DIR = "/kaggle/input/vlsp2025-kd-wheels"

# if os.path.exists(WHEELS_DIR):
#     wheel_files = list(Path(WHEELS_DIR).glob("*.whl"))
#     print(f"Installing {len(wheel_files)} wheels from offline cache...")
#     result = subprocess.run(
#         [sys.executable, "-m", "pip", "install", "-q",
#          "--no-index", "--find-links", WHEELS_DIR,
#          "transformers", "peft", "accelerate", "datasets",
#          "bitsandbytes", "trl", "safetensors", "sentencepiece",
#          "protobuf", "sympy", "pyyaml", "pyarrow", "tqdm", "pandas",
#          "huggingface_hub", "tokenizers"],
#         capture_output=True, text=True,
#     )
#     if result.returncode == 0:
#         print("Offline installation complete.")
#     else:
#         print(f"Offline wheels failed, falling back to pip:\n{result.stderr[-300:]}")
#         subprocess.run([sys.executable, "-m", "pip", "install", "-q",
#             "transformers>=5.0", "peft>=0.18", "accelerate>=1.0",
#             "datasets>=4.0", "bitsandbytes>=0.49", "trl>=1.0",
#             "safetensors", "sentencepiece", "protobuf", "sympy"], check=False)
# else:
#     print("Wheels not available, installing from pip (requires internet)...")
#     subprocess.run([sys.executable, "-m", "pip", "install", "-q",
#         "transformers>=5.0", "peft>=0.18", "accelerate>=1.0",
#         "datasets>=4.0", "bitsandbytes>=0.49", "trl>=1.0",
#         "safetensors", "sentencepiece", "protobuf", "sympy"], check=False)
#     print("pip installation complete.")

# # Flash Attention (optional — RTX/A100/H100 only, not P100/T4)
# try:
#     subprocess.run([sys.executable, "-m", "pip", "install", "-q",
#                     "flash-attn", "--no-build-isolation"],
#                    check=True, capture_output=True, timeout=300)
#     print("Flash Attention installed.")
# except Exception:
#     print("Flash Attention not available (optional, continuing without it).")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 2: Setup Working Directory, sys.path, and Dataset Links
# # ═══════════════════════════════════════════════════════════════════════
# import os, sys, shutil
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# WORK_DIR.mkdir(parents=True, exist_ok=True)

# # Step 1: Register WORK_DIR on sys.path BEFORE copying (slot is created now)
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))

# # Step 2: Copy pipeline source code into WORK_DIR
# CODE_SRC = Path("/kaggle/input/vlsp2025-kd-pipeline")
# if not CODE_SRC.exists():
#     raise FileNotFoundError(
#         f"Pipeline code not found at {CODE_SRC}\n"
#         "Add dataset 'thanhduc1108/vlsp2025-kd-pipeline' as notebook input."
#     )

# for d in ["pipeline", "src", "configs"]:
#     src = CODE_SRC / d
#     dst = WORK_DIR / d
#     if src.exists():
#         if dst.exists():
#             shutil.rmtree(dst)
#         shutil.copytree(src, dst)
#         n = sum(1 for _ in dst.rglob("*") if _.is_file())
#         print(f"  Copied {d}/ ({n} files)")
#     else:
#         print(f"  WARNING: {d}/ missing from pipeline dataset")

# if (CODE_SRC / "requirements.txt").exists():
#     shutil.copy2(CODE_SRC / "requirements.txt", WORK_DIR / "requirements.txt")

# # Step 3: Verify the copy succeeded and pipeline is importable
# pipeline_init = WORK_DIR / "pipeline" / "__init__.py"
# if not pipeline_init.exists():
#     raise FileNotFoundError(
#         f"pipeline/__init__.py not found after copy.\n"
#         "The vlsp2025-kd-pipeline dataset may be outdated. Re-upload with:\n"
#         "  python scripts/kaggle_upload.py --upload-code"
#     )
# print(f"\nPipeline code ready at: {WORK_DIR / 'pipeline'}")

# # Step 4: Set working directory so relative paths work
# os.chdir(WORK_DIR)

# # Step 5: Symlink datasets into expected locations
# DATASET_DIR = WORK_DIR / "dataset"
# DATASET_DIR.mkdir(parents=True, exist_ok=True)

# for src_path, dst_name, label in [
#     ("/kaggle/input/vinumericalqa-private", "viNumericalQA_private", "ViNumQA"),
#     ("/kaggle/input/finqa-en",              "finqa_en",              "FinQA"),
# ]:
#     src = Path(src_path)
#     dst = DATASET_DIR / dst_name
#     if src.exists():
#         if not dst.exists():
#             dst.symlink_to(src)
#         n = sum(1 for _ in src.rglob("*") if _.is_file())
#         print(f"  {label} linked ({n} files): {dst}")
#     else:
#         print(f"  WARNING: {label} not found at {src_path}")

# print(f"\nWorking directory : {WORK_DIR}")
# print(f"sys.path[0]       : {sys.path[0]}")
# print(f"Contents          : {[p.name for p in sorted(WORK_DIR.iterdir())]}")

# # Quick import smoke-test
# try:
#     import importlib
#     spec = importlib.util.spec_from_file_location(
#         "pipeline.config", str(WORK_DIR / "pipeline" / "config.py"))
#     print("Pipeline import check: OK")
# except Exception as e:
#     print(f"Pipeline import check FAILED: {e}")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 3: Verify Environment & Detect GPU Profile
# # ═══════════════════════════════════════════════════════════════════════
# import sys
# from pathlib import Path

# # Idempotent sys.path guard (safe even if cells run out of order)
# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))

# import torch

# print(f"PyTorch     : {torch.__version__}")
# print(f"CUDA        : {torch.cuda.is_available()}")

# if not torch.cuda.is_available():
#     raise RuntimeError(
#         "No GPU detected!\n"
#         "Enable GPU: Notebook -> Settings -> Accelerator -> GPU T4 / P100 / RTX 6000"
#     )

# gpu_name = torch.cuda.get_device_name(0)
# vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
# print(f"GPU         : {gpu_name}  ({vram_gb:.1f} GB VRAM)")

# import transformers, peft, accelerate, datasets
# print(f"Transformers: {transformers.__version__}")
# print(f"PEFT        : {peft.__version__}")
# print(f"Accelerate  : {accelerate.__version__}")
# print(f"Datasets    : {datasets.__version__}")

# try:
#     import flash_attn
#     HAS_FLASH = True
#     print(f"Flash Attn  : {flash_attn.__version__}")
# except ImportError:
#     HAS_FLASH = False
#     print("Flash Attn  : not available")

# if "6000" in gpu_name or vram_gb > 90:
#     GPU_PROFILE = "rtx6000_96gb"
# elif "A100" in gpu_name and vram_gb > 70:
#     GPU_PROFILE = "a100_80gb"
# elif "P100" in gpu_name or vram_gb < 20:
#     GPU_PROFILE = "p100_16gb"
# else:
#     GPU_PROFILE = "p100_16gb"

# print(f"\nGPU Profile : {GPU_PROFILE}")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 4: Resolve Model Paths (Kaggle offline -> HuggingFace fallback)
# # ═══════════════════════════════════════════════════════════════════════
# import sys
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))

# TEACHER_MODEL_ID = "Qwen/Qwen3.5-27B"
# STUDENT_MODEL_ID = "Qwen/Qwen3.5-4B"


# def resolve_model_path(model_id: str) -> str:
#     """Check /kaggle/input/* and /kaggle/models/* for weights; fall back to HF id."""
#     base_name = model_id.split("/")[-1]
#     slugs = set()
#     for name in [base_name, base_name.lower()]:
#         slugs.add(name)
#         slugs.add(name.replace(".", "-").replace("_", "-"))
#         slugs.add(name.replace(".", "_").replace("-", "_"))

#     for root in [Path("/kaggle/input"), Path("/kaggle/models")]:
#         if not root.exists():
#             continue
#         for entry in root.iterdir():
#             if entry.name.lower() in {s.lower() for s in slugs}:
#                 for candidate in [entry, *entry.rglob("config.json")]:
#                     check_dir = candidate if candidate.is_dir() else candidate.parent
#                     has_weights = (
#                         any(check_dir.glob("*.safetensors")) or
#                         any(check_dir.glob("*.bin")) or
#                         (check_dir / "config.json").exists()
#                     )
#                     if has_weights:
#                         print(f"  Found offline: {check_dir}")
#                         return str(check_dir)

#     print(f"  Not found offline -> using HuggingFace: {model_id}")
#     return model_id


# print("Resolving teacher model...")
# TEACHER_PATH = resolve_model_path(TEACHER_MODEL_ID)
# print("Resolving student model...")
# STUDENT_PATH = resolve_model_path(STUDENT_MODEL_ID)
# print(f"\nTeacher: {TEACHER_PATH}")
# print(f"Student: {STUDENT_PATH}")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 5: Configure Pipeline
# # ═══════════════════════════════════════════════════════════════════════
# import os, sys
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.config import load_config, save_config

# config_overrides = {
#     "model": {
#         "teacher_model": TEACHER_PATH,
#         "student_model": STUDENT_PATH,
#         "use_flash_attention": HAS_FLASH,
#     },
#     "data": {
#         "vinumqa_train":        "dataset/viNumericalQA_private/train.json",
#         "vinumqa_valid":        "dataset/viNumericalQA_private/valid.json",
#         "vinumqa_test":         "dataset/viNumericalQA_private/test.json",
#         "vinumqa_private_test": "dataset/viNumericalQA_private/private_test.json",
#         "finqa_dir":            "dataset/finqa_en",
#     },
# }

# cfg = load_config(gpu_profile=GPU_PROFILE, overrides=config_overrides)

# print(f"\n{'='*60}")
# print("Pipeline Configuration")
# print(f"{'='*60}")
# print(f"  GPU Profile   : {GPU_PROFILE}")
# print(f"  Teacher model : {cfg.model.teacher_model}")
# print(f"  Student model : {cfg.model.student_model}")
# print(f"  Teacher quant : {cfg.model.teacher_quantization}")
# print(f"  Flash Attn    : {cfg.model.use_flash_attention}")
# print(f"  dtype         : {cfg.model.torch_dtype}")
# print(f"  SFT epochs    : {cfg.sft.num_epochs}  |  LoRA r={cfg.sft.lora_r}")
# print(f"  GRPO epochs   : {cfg.grpo.num_epochs}  |  N generations={cfg.grpo.num_generations}")
# print(f"  Inference N   : {cfg.inference.num_candidates}")
# print(f"{'='*60}")

# save_config(cfg, str(WORK_DIR / "data/pipeline/config.yaml"))
# print("Config saved.")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 6: Phase 1 — Data Preparation
# # ═══════════════════════════════════════════════════════════════════════
# import os, sys, time
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.data_prep import run_data_prep

# print("=" * 60)
# print("PHASE 1: DATA PREPARATION")
# print("=" * 60)

# t0 = time.time()
# data_paths = run_data_prep(cfg)
# print(f"\nData prep completed in {time.time()-t0:.1f}s")
# for k, v in data_paths.items():
#     print(f"  {k}: {v}")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 7: Phase 2 — Teacher Distillation
# # ═══════════════════════════════════════════════════════════════════════
# import gc, os, sys, time, torch
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.teacher_distill import run_teacher_distillation

# print("=" * 60)
# print("PHASE 2: TEACHER DISTILLATION")
# print("=" * 60)

# t0 = time.time()
# distilled_path = run_teacher_distillation(cfg, data_paths["teacher_input"])
# print(f"\nTeacher distillation completed in {time.time()-t0:.1f}s")
# print(f"  Distilled SFT data: {distilled_path}")

# gc.collect()
# torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 8: Phase 3 — SFT Training
# # ═══════════════════════════════════════════════════════════════════════
# import gc, os, sys, time, torch
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.train_sft import run_sft_training

# print("=" * 60)
# print("PHASE 3: SUPERVISED FINE-TUNING (SFT)")
# print("=" * 60)

# t0 = time.time()
# sft_model_path = run_sft_training(
#     cfg,
#     train_path=distilled_path,
#     valid_path=data_paths["sft_valid"],
# )
# print(f"\nSFT training completed in {time.time()-t0:.1f}s")
# print(f"  SFT model: {sft_model_path}")

# gc.collect()
# torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 9: Phase 4 — GRPO Training
# # (Comment out / skip to use SFT-only: set grpo_model_path = sft_model_path)
# # ═══════════════════════════════════════════════════════════════════════
# import gc, os, sys, time, torch
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.train_grpo import run_grpo_training

# print("=" * 60)
# print("PHASE 4: GRPO WITH PCPO REWARD")
# print("=" * 60)

# t0 = time.time()
# grpo_model_path = run_grpo_training(cfg, sft_model_path)
# print(f"\nGRPO training completed in {time.time()-t0:.1f}s")
# print(f"  GRPO model: {grpo_model_path}")

# gc.collect()
# torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 10: Phase 5 — Inference with Majority Voting
# # ═══════════════════════════════════════════════════════════════════════
# import gc, json, os, sys, time, torch
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.inference import run_inference

# print("=" * 60)
# print("PHASE 5: MULTI-PATH INFERENCE + MAJORITY VOTING")
# print("=" * 60)

# # Use best available model: GRPO > SFT
# final_model = grpo_model_path if Path(grpo_model_path).exists() else sft_model_path
# print(f"Using model : {final_model}")

# # data_paths["sft_valid"] was produced by run_data_prep — correct SFT format for inference
# test_input_path = data_paths["sft_valid"]
# print(f"Test input  : {test_input_path}")

# t0 = time.time()
# predictions_path = run_inference(cfg, final_model, test_input_path)
# print(f"\nInference completed in {time.time()-t0:.1f}s")
# print(f"  Predictions: {predictions_path}")

# gc.collect()
# torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 11: Phase 6 — Evaluation (EA + PA)
# # ═══════════════════════════════════════════════════════════════════════
# import gc, os, sys, torch
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.evaluate import evaluate_against_dataset

# print("=" * 60)
# print("PHASE 6: EVALUATION (EA + PA)")
# print("=" * 60)

# # Original ViNumQA valid JSON supplies table data needed for proper EA/PA
# test_dataset_path = str(WORK_DIR / "dataset/viNumericalQA_private/valid.json")
# eval_output       = str(WORK_DIR / "data/pipeline/eval_results.json")

# results = evaluate_against_dataset(predictions_path, test_dataset_path, eval_output)

# gc.collect()
# torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 12: Baseline Comparison — Zero-shot Student Model (optional)
# # Skip this cell if you only need KD results.
# # ═══════════════════════════════════════════════════════════════════════
# import gc, os, shutil, sys, torch
# from pathlib import Path

# WORK_DIR = Path("/kaggle/working/vlsp2025")
# if str(WORK_DIR) not in sys.path:
#     sys.path.insert(0, str(WORK_DIR))
# os.chdir(WORK_DIR)

# from pipeline.inference import run_inference
# from pipeline.evaluate import evaluate_against_dataset

# pipeline_out = WORK_DIR / "data/pipeline"

# # run_inference always writes to predictions.json — back up KD results first
# kd_backup = str(pipeline_out / "predictions_kd.json")
# shutil.copy2(predictions_path, kd_backup)
# print(f"KD predictions backed up -> {kd_backup}")

# print(f"\nRunning zero-shot baseline with: {STUDENT_PATH}")
# run_inference(cfg, model_path=STUDENT_PATH, test_data_path=test_input_path)

# # Rename baseline output, restore KD predictions
# baseline_pred_path = str(pipeline_out / "predictions_baseline.json")
# shutil.copy2(str(pipeline_out / "predictions.json"), baseline_pred_path)
# shutil.copy2(kd_backup, str(pipeline_out / "predictions.json"))
# print(f"Baseline predictions saved -> {baseline_pred_path}")

# test_dataset_path = str(WORK_DIR / "dataset/viNumericalQA_private/valid.json")
# baseline_results = evaluate_against_dataset(
#     baseline_pred_path,
#     test_dataset_path,
#     str(pipeline_out / "eval_baseline.json"),
# )

# gc.collect()
# torch.cuda.empty_cache()


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 13: Results Summary
# # ═══════════════════════════════════════════════════════════════════════
# print("\n" + "=" * 60)
# print("FINAL RESULTS SUMMARY")
# print("=" * 60)

# print(f"\n{'Model':<30} {'EA':>8} {'PA':>8} {'Valid%':>8}")
# print("-" * 60)
# print(f"{'Baseline (zero-shot)':<30} "
#       f"{baseline_results['execution_accuracy']:>7.2%} "
#       f"{baseline_results['program_accuracy']:>7.2%} "
#       f"{baseline_results['valid_rate']:>7.2%}")
# print(f"{'After KD (SFT+GRPO)':<30} "
#       f"{results['execution_accuracy']:>7.2%} "
#       f"{results['program_accuracy']:>7.2%} "
#       f"{results['valid_rate']:>7.2%}")

# ea_delta = results['execution_accuracy'] - baseline_results['execution_accuracy']
# pa_delta = results['program_accuracy']   - baseline_results['program_accuracy']
# print(f"\nImprovement: EA {ea_delta:+.2%},  PA {pa_delta:+.2%}")


# # ═══════════════════════════════════════════════════════════════════════
# # CELL 14: Save All Outputs
# # ═══════════════════════════════════════════════════════════════════════
# import json, os, shutil, sys
# from pathlib import Path

# WORK_DIR     = Path("/kaggle/working/vlsp2025")
# output_dir   = WORK_DIR / "outputs"
# pipeline_out = WORK_DIR / "data/pipeline"
# output_dir.mkdir(parents=True, exist_ok=True)

# if Path(final_model).exists():
#     final_model_output = output_dir / "final_model"
#     if final_model_output.exists():
#         shutil.rmtree(final_model_output)
#     shutil.copytree(final_model, final_model_output)
#     print(f"Final model saved -> {final_model_output}")

# for fname in ["eval_results.json", "eval_baseline.json",
#               "predictions.json", "predictions_kd.json", "predictions_baseline.json",
#               "config.yaml"]:
#     src = pipeline_out / fname
#     if src.exists():
#         shutil.copy2(src, output_dir / fname)
#         print(f"  Copied {fname}")

# summary = {
#     "gpu_profile":   GPU_PROFILE,
#     "teacher_model": TEACHER_MODEL_ID,
#     "student_model": STUDENT_MODEL_ID,
#     "baseline": {
#         "EA":         baseline_results["execution_accuracy"],
#         "PA":         baseline_results["program_accuracy"],
#         "valid_rate": baseline_results["valid_rate"],
#     },
#     "after_kd": {
#         "EA":         results["execution_accuracy"],
#         "PA":         results["program_accuracy"],
#         "valid_rate": results["valid_rate"],
#     },
#     "improvement": {"EA_delta": ea_delta, "PA_delta": pa_delta},
# }
# with open(output_dir / "summary.json", "w") as f:
#     json.dump(summary, f, indent=2, ensure_ascii=False)

# print(f"\nAll outputs -> {output_dir}")
# print(f"Files: {[p.name for p in sorted(output_dir.iterdir())]}")
# print("\nDone! Check /kaggle/working/vlsp2025/outputs/ for final results.")


In [ ]:
# """
# VLSP 2025 Baseline - Kaggle Notebook (Offline Execution)
# =========================================================
# This script is designed to run on Kaggle with RTX 6000 Pro 96GB
# WITHOUT internet access.

# Setup requirements:
# 1. Upload the offline package as a Kaggle Dataset (e.g., 'vlsp2025-offline')
# 2. Add the dataset to your notebook
# 3. GPU: RTX 6000 Pro 96GB
# 4. Internet: OFF

# The offline package should contain:
#   /kaggle/input/vlsp2025-offline/
#     ├── wheels/        (pip wheels for offline install)
#     ├── models/        (pre-downloaded HF model weights)
#     ├── data/receive/  (ViNumQA dataset)
#     └── code/          (project source code)
# """

# import os
# import sys
# import subprocess
# import json
# import time
# import gc
# from pathlib import Path

# # ============================================================================
# # CONFIGURATION - Edit these paths based on your Kaggle dataset name
# # ============================================================================

# # Path to the offline package dataset on Kaggle
# OFFLINE_DATASET = "/kaggle/input/vlsp2025-offline"

# # Working directory
# WORK_DIR = "/kaggle/working"

# # Models to run (comment out models you don't want to test)
# MODELS_TO_RUN = [
#     "qwen3.5-4b",
#     "qwen3.5-9b",
#     "qwen3.5-27b",
#     "qwen3.5-35b-a3b",
#     "qwen3.5-122b-a10b",   # requires 4-bit quantization
# ]

# # Max samples per model (None = all 497 public test samples)
# MAX_SAMPLES = None  # Set to e.g. 10 for quick test

# # ============================================================================
# # STEP 1: Install dependencies offline
# # ============================================================================

# print("=" * 60)
# print(" Step 1: Installing offline dependencies")
# print("=" * 60)

# wheels_dir = os.path.join(OFFLINE_DATASET, "wheels")
# if os.path.exists(wheels_dir) and os.listdir(wheels_dir):
#     subprocess.run([
#         sys.executable, "-m", "pip", "install",
#         "--no-index", "--find-links", wheels_dir,
#         "transformers", "accelerate", "peft", "bitsandbytes",
#         "datasets", "pandas", "tqdm", "pyarrow", "pyyaml",
#         "sentencepiece", "protobuf", "safetensors",
#         "huggingface_hub", "tokenizers",
#     ], check=False, capture_output=True)
#     print("  Offline packages installed.")
# else:
#     print("  No wheels directory found. Using pre-installed packages.")
#     print("  If packages are missing, the script will fail at import time.")

# # Try installing flash-attn (may fail on some environments)
# try:
#     subprocess.run([
#         sys.executable, "-m", "pip", "install",
#         "--no-index", "--find-links", wheels_dir,
#         "flash-attn",
#     ], check=True, capture_output=True, timeout=120)
#     print("  flash-attn installed.")
# except Exception:
#     print("  flash-attn not available (will use default attention).")


# # ============================================================================
# # STEP 2: Setup project code
# # ============================================================================

# print("\n" + "=" * 60)
# print(" Step 2: Setting up project code")
# print("=" * 60)

# code_dir = os.path.join(OFFLINE_DATASET, "code")
# if os.path.exists(code_dir):
#     # Copy code to working dir for write access
#     os.makedirs(WORK_DIR, exist_ok=True)
#     os.system(f"cp -r {code_dir}/* {WORK_DIR}/")
#     print(f"  Code copied to {WORK_DIR}")
# else:
#     print(f"  WARNING: Code directory not found at {code_dir}")
#     print("  Attempting to use current directory...")

# sys.path.insert(0, WORK_DIR)
# os.chdir(WORK_DIR)

# # Setup data directory
# data_src = os.path.join(OFFLINE_DATASET, "data", "receive")
# data_dst = os.path.join(WORK_DIR, "data", "receive")
# os.makedirs(data_dst, exist_ok=True)
# if os.path.exists(data_src):
#     os.system(f"cp -n {data_src}/*.json {data_dst}/")
#     print(f"  Data copied to {data_dst}")

# # Verify
# for f in ["train.json", "valid.json", "test.json"]:
#     p = os.path.join(data_dst, f)
#     if os.path.exists(p):
#         with open(p) as fh:
#             print(f"    {f}: {len(json.load(fh))} samples")

# # ============================================================================
# # STEP 3: Verify environment
# # ============================================================================

# print("\n" + "=" * 60)
# print(" Step 3: Environment verification")
# print("=" * 60)

# import torch
# print(f"  PyTorch: {torch.__version__}")
# print(f"  CUDA: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"  GPU: {torch.cuda.get_device_name(0)}")
#     vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
#     print(f"  VRAM: {vram:.1f} GB")

# import transformers
# print(f"  Transformers: {transformers.__version__}")

# try:
#     import peft
#     print(f"  PEFT: {peft.__version__}")
# except ImportError:
#     print("  PEFT: NOT INSTALLED")

# try:
#     import bitsandbytes
#     print(f"  BitsAndBytes: {bitsandbytes.__version__}")
# except ImportError:
#     print("  BitsAndBytes: NOT INSTALLED (4-bit quant unavailable)")

# # Check flash attention
# HAS_FLASH_ATTN = False
# try:
#     import flash_attn
#     HAS_FLASH_ATTN = True
#     print(f"  Flash Attention: {flash_attn.__version__}")
# except ImportError:
#     print("  Flash Attention: NOT INSTALLED (using eager attention)")


# # ============================================================================
# # STEP 4: Model path resolution
# # ============================================================================

# print("\n" + "=" * 60)
# print(" Step 4: Resolving model paths")
# print("=" * 60)

# models_dir = os.path.join(OFFLINE_DATASET, "models")

# # Map from model_id to local path
# MODEL_LOCAL_PATHS = {}
# MODEL_CONFIGS = {
#     "qwen3.5-4b": {
#         "model_id": "Qwen/Qwen3.5-4B",
#         "quantization": None,
#         "torch_dtype": "bfloat16",
#         "max_seq_length": 16384,
#         "max_new_tokens": 2048,
#         "num_candidates": 15,
#         "estimated_vram_gb": 9,
#     },
#     "qwen3.5-9b": {
#         "model_id": "Qwen/Qwen3.5-9B",
#         "quantization": None,
#         "torch_dtype": "bfloat16",
#         "max_seq_length": 16384,
#         "max_new_tokens": 2048,
#         "num_candidates": 15,
#         "estimated_vram_gb": 19,
#     },
#     "qwen3.5-27b": {
#         "model_id": "Qwen/Qwen3.5-27B",
#         "quantization": None,
#         "torch_dtype": "bfloat16",
#         "max_seq_length": 12288,
#         "max_new_tokens": 2048,
#         "num_candidates": 10,
#         "estimated_vram_gb": 55,
#     },
#     "qwen3.5-35b-a3b": {
#         "model_id": "Qwen/Qwen3.5-35B-A3B",
#         "quantization": None,
#         "torch_dtype": "bfloat16",
#         "max_seq_length": 16384,
#         "max_new_tokens": 2048,
#         "num_candidates": 15,
#         "estimated_vram_gb": 72,
#     },
#     "qwen3.5-122b-a10b": {
#         "model_id": "Qwen/Qwen3.5-122B-A10B",
#         "quantization": "4bit",
#         "torch_dtype": "bfloat16",
#         "max_seq_length": 12288,
#         "max_new_tokens": 2048,
#         "num_candidates": 10,
#         "estimated_vram_gb": 70,
#     },
# }

# for key, cfg in MODEL_CONFIGS.items():
#     model_id = cfg["model_id"]
#     # Check offline package first
#     local_name = model_id.replace("/", "_")
#     local_path = os.path.join(models_dir, local_name)

#     if os.path.exists(local_path) and os.listdir(local_path):
#         MODEL_LOCAL_PATHS[key] = local_path
#         size_gb = sum(f.stat().st_size for f in Path(local_path).rglob('*') if f.is_file()) / 1e9
#         print(f"  {key}: {local_path} ({size_gb:.1f} GB)")
#     else:
#         # Try Kaggle HF cache
#         hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
#         print(f"  {key}: Will use HF model_id '{model_id}' (not found locally)")
#         MODEL_LOCAL_PATHS[key] = model_id

# # ============================================================================
# # STEP 5: Run Baselines
# # ============================================================================

# print("\n" + "=" * 60)
# print(" Step 5: Running baselines")
# print("=" * 60)

# import re
# from collections import Counter
# from tqdm import tqdm

# # Import project modules
# from pipeline.program_executor import execute_program, format_answer, validate_program
# from pipeline.data_prep import table_to_markdown
# from src.assets.template import prompt as prompt_template


# def build_prompt(sample):
#     pre_text = " ".join(sample.get("pre_text", []))
#     table_md = table_to_markdown(sample.get("table", []))
#     post_text = " ".join(sample.get("post_text", []))
#     question = sample["qa"]["question"]
#     text = prompt_template
#     text = text.replace("pre_text_placeholder", pre_text)
#     text = text.replace("table_placeholder", table_md)
#     text = text.replace("post_text_placeholder", post_text)
#     text = text.replace("question_placeholder", question)
#     return text


# def extract_program_and_answer(text):
#     prog_matches = list(re.finditer(
#         r"\*\*Chương trình tính toán:\*\*\s*((?:.|\n)*?)(?=\s*\*\*|$)", text
#     ))
#     prog = prog_matches[-1].group(1).strip() if prog_matches else None
#     ans_matches = list(re.finditer(
#         r"\*\*Đáp án cuối cùng:\*\*\s*((?:.|\n)*?)(?=\s*\*\*|$)", text
#     ))
#     ans = ans_matches[-1].group(1).strip() if ans_matches else None
#     return prog, ans


# def majority_vote(candidates, table=None):
#     answers, programs = [], []
#     for text in candidates:
#         prog, ans = extract_program_and_answer(text)
#         if prog and validate_program(prog):
#             result = execute_program(prog, table)
#             if result is not None:
#                 answers.append(format_answer(result))
#                 programs.append(prog)
#             elif ans:
#                 answers.append(ans.strip())
#                 programs.append(prog)
#         elif ans:
#             answers.append(ans.strip())
#             programs.append(prog or "")

#     if not answers:
#         return {"program": None, "answer": None, "confidence": 0.0, "num_valid": 0}

#     counter = Counter(answers)
#     best_answer, best_count = counter.most_common(1)[0]
#     best_program = next((p for a, p in zip(answers, programs) if a == best_answer), None)
#     return {
#         "program": best_program, "answer": best_answer,
#         "confidence": best_count / len(candidates), "num_valid": len(answers),
#     }


# def run_model_baseline(model_key, test_data, max_samples=None):
#     """Run baseline for a single model."""
#     from transformers import AutoModelForCausalLM, AutoTokenizer

#     cfg = MODEL_CONFIGS[model_key]
#     model_path = MODEL_LOCAL_PATHS[model_key]

#     print(f"\n{'#' * 60}")
#     print(f"# Model: {cfg['model_id']}")
#     print(f"# Path: {model_path}")
#     print(f"# Quantization: {cfg['quantization'] or 'none'}")
#     print(f"# Candidates: {cfg['num_candidates']}")
#     print(f"{'#' * 60}\n")

#     if max_samples:
#         test_data = test_data[:max_samples]

#     # Model loading
#     dtype = torch.bfloat16 if cfg["torch_dtype"] == "bfloat16" else torch.float16
#     model_kwargs = {
#         "trust_remote_code": True,
#         "torch_dtype": dtype,
#         "device_map": "auto",
#     }

#     if cfg["quantization"] == "4bit":
#         from transformers import BitsAndBytesConfig
#         model_kwargs["quantization_config"] = BitsAndBytesConfig(
#             load_in_4bit=True, bnb_4bit_compute_dtype=dtype,
#             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
#         )

#     if HAS_FLASH_ATTN:
#         model_kwargs["attn_implementation"] = "flash_attention_2"

#     print(f"Loading model...")
#     tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
#     model = AutoModelForCausalLM.from_pretrained(model_path, **model_kwargs)
#     model.eval()

#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token

#     if torch.cuda.is_available():
#         vram = torch.cuda.memory_allocated() / 1024**3
#         print(f"  VRAM used: {vram:.1f} GB")

#     # Inference
#     predictions = []
#     start = time.time()
#     n_cand = cfg["num_candidates"]

#     for i, sample in enumerate(tqdm(test_data, desc=model_key)):
#         prompt = build_prompt(sample)
#         table = sample.get("table")

#         messages = [{"role": "user", "content": prompt}]
#         input_text = tokenizer.apply_chat_template(
#             messages, tokenize=False, add_generation_prompt=True
#         )
#         inputs = tokenizer(
#             input_text, return_tensors="pt",
#             truncation=True, max_length=cfg["max_seq_length"]
#         )
#         inputs = {k: v.to(model.device) for k, v in inputs.items()}

#         candidates = []
#         for _ in range(n_cand):
#             with torch.no_grad():
#                 outputs = model.generate(
#                     **inputs,
#                     max_new_tokens=cfg["max_new_tokens"],
#                     temperature=0.7, top_p=0.95,
#                     do_sample=True,
#                     pad_token_id=tokenizer.pad_token_id,
#                 )
#             new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
#             text = tokenizer.decode(new_tokens, skip_special_tokens=True)
#             candidates.append(text)

#         vote = majority_vote(candidates, table)

#         predictions.append({
#             "id": sample["id"],
#             "predicted_program": vote["program"],
#             "predicted_answer": vote["answer"],
#             "confidence": vote["confidence"],
#             "num_valid": vote["num_valid"],
#             "gold_program": sample["qa"].get("program", ""),
#             "gold_answer": str(sample["qa"].get("exe_ans", "")),
#         })

#     elapsed = time.time() - start
#     print(f"  Done: {elapsed:.0f}s ({elapsed/len(test_data):.1f}s/sample)")

#     # Cleanup
#     del model, tokenizer
#     gc.collect()
#     torch.cuda.empty_cache()
#     time.sleep(2)

#     return predictions, elapsed


# # Load test data
# test_path = os.path.join(WORK_DIR, "data", "receive", "test.json")
# with open(test_path, "r", encoding="utf-8") as f:
#     test_data = json.load(f)
# print(f"\nPublic test: {len(test_data)} samples")

# # Run each model
# all_results = []
# output_dir = os.path.join(WORK_DIR, "outputs", "baseline")
# os.makedirs(output_dir, exist_ok=True)

# for model_key in MODELS_TO_RUN:
#     if model_key not in MODEL_CONFIGS:
#         print(f"  SKIP unknown model: {model_key}")
#         continue

#     try:
#         predictions, elapsed = run_model_baseline(model_key, test_data, MAX_SAMPLES)

#         # Save predictions
#         pred_path = os.path.join(output_dir, f"predictions_{model_key}.json")
#         with open(pred_path, "w", encoding="utf-8") as f:
#             json.dump(predictions, f, ensure_ascii=False, indent=2)

#         # Evaluate
#         total = len(predictions)
#         ea_ok = sum(1 for p in predictions if _ea_match(p))
#         pa_ok = sum(1 for p in predictions if _pa_match(p))
#         valid = sum(1 for p in predictions if p["predicted_program"] and
#                     validate_program(p["predicted_program"]))

#         result = {
#             "model": model_key,
#             "total": total,
#             "ea": ea_ok / total if total else 0,
#             "pa": pa_ok / total if total else 0,
#             "valid_rate": valid / total if total else 0,
#             "elapsed": elapsed,
#         }
#         all_results.append(result)

#         print(f"  EA: {result['ea']:.2%} | PA: {result['pa']:.2%} | Valid: {result['valid_rate']:.2%}")

#     except Exception as e:
#         print(f"  ERROR: {model_key} - {e}")
#         import traceback
#         traceback.print_exc()
#         all_results.append({"model": model_key, "error": str(e)})


# def _ea_match(pred):
#     try:
#         p = float(pred.get("predicted_answer", ""))
#         g = float(pred.get("gold_answer", ""))
#         return abs(p - g) / max(abs(g), 1e-10) < 1e-4 if g != 0 else abs(p - g) < 1e-5
#     except (ValueError, TypeError):
#         return (pred.get("predicted_answer") or "").strip() == (pred.get("gold_answer") or "").strip()


# def _pa_match(pred):
#     from pipeline.evaluate import programs_match
#     return programs_match(pred.get("predicted_program", ""), pred.get("gold_program", ""))


# # ============================================================================
# # STEP 6: Summary
# # ============================================================================

# print("\n" + "=" * 80)
# print(" BASELINE RESULTS SUMMARY")
# print("=" * 80)
# print(f" {'Model':<25} {'EA':>8} {'PA':>8} {'Valid%':>8} {'Time':>10}")
# print(f" {'-'*25} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")
# for r in all_results:
#     if "error" in r:
#         print(f" {r['model']:<25} {'ERROR':>8}")
#     else:
#         print(f" {r['model']:<25} {r['ea']:>7.2%} {r['pa']:>7.2%} "
#               f"{r['valid_rate']:>7.2%} {r['elapsed']:>8.0f}s")
# print("=" * 80)

# # Save summary
# summary_path = os.path.join(output_dir, "baseline_summary.json")
# with open(summary_path, "w", encoding="utf-8") as f:
#     json.dump(all_results, f, ensure_ascii=False, indent=2)
# print(f"\nSummary saved: {summary_path}")

# # Save outputs to /kaggle/working for download
# print("\nAll outputs saved to /kaggle/working/outputs/baseline/")
# print("Download from Kaggle Output tab after notebook completes.")

In [ ]:
# # Chạy dòng này trong cell (dùng dấu chấm than ! ở đầu)
# # Thêm --provider github để bỏ qua bước chọn menu

# !curl -Lk 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' --output vscode_cli.tar.gz
# !tar -xf vscode_cli.tar.gz
# !./code tunnel user login --provider github
# # !./code tunnel --accept-server-license-terms --name KaggleTraining

In [ ]:
# # 2. Khởi động tunnel (lúc này đã có file cache đăng nhập)
# !./code tunnel --accept-server-license-terms --name KaggleTraining